> ## MODEL v3 (CLIP ViT-B-32 frozen) -- pembanding, TIDAK live di backend
>
> Backend sekarang tetap pakai model v2 (`convnext_small` fine-tuned, 22
> kelas) -- lihat `notebooks/README.md`. v3 di sini HANYA pembanding utk
> presentasi ("apakah CLIP frozen + linear probe kompetitif dgn fine-tuning
> penuh?"), BUKAN kandidat promosi otomatis ke root.
>
> Index katalog ditulis ke
> `data/cache/v3_clip/catalog_embeddings_clip.npz` -- terpisah total dari
> `data/cache/catalog_embeddings.npz` (live, dari v2) maupun
> `data/cache/v1_11class_backup/catalog_embeddings.npz` (v1). Prasyarat:
> `01_train.ipynb` di folder ini sudah dijalankan (untuk checkpoint
> classifier head -- dipakai section 6 "prediksi style", opsional).

# 02 — Embedding & Evaluasi Retrieval -- MODEL v3 (CLIP)

Sama seperti `notebooks/v2_22class/02_embedding_retrieval.ipynb`, tapi
embedding diambil langsung dari CLIP `encode_image` (frozen) -- TIDAK dari
`forward_features` classifier (itu khusus arsitektur `GaleriaArtNet` ConvNeXt
di `src/model.py`, tidak dipakai di sini).

1. Ekstrak embedding CLIP: gallery (train+val) vs query (test).
2. Recall@k / Precision@k / mAP@k vs baseline acak.
3. Contoh 1 query -> top-5 hasil.
4. Index embedding seluruh katalog (`data/cache/v3_clip/catalog_embeddings_clip.npz`).
5. Instance retrieval ("scan lukisan") -- simulasi foto HP, cek apakah CLIP
   menemukan file ASLI-nya di katalog.
6. Catatan interpretasi + perbandingan ke v1/v2.

## 1. Setup

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml-visual-search/ (berisi src/ & configs/) tidak ketemu")
    _p = _p.parent
os.chdir(_p)
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import open_clip

print("working dir:", os.getcwd())
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
from src.utils import load_config, set_seed, get_device, resolve_path
from src.dataset import build_label_mapping, WikiArtDataset
from src.embedding import save_embedding_index, nearest_neighbors, evaluate_retrieval, chance_recall_at_k
from torch.utils.data import DataLoader

cfg = load_config("configs/config.yaml")
clip_cfg = cfg["baseline_clip"]
set_seed(cfg["seed"])
device = get_device(cfg["device"])

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    clip_cfg["model_name"], pretrained=clip_cfg["pretrained"]
)
clip_model = clip_model.to(device).eval()
for p in clip_model.parameters():
    p.requires_grad_(False)

label_to_idx = build_label_mapping(cfg)
idx_to_label = {v: k for k, v in label_to_idx.items()}
num_classes = len(label_to_idx)
print(f"CLIP {clip_cfg['model_name']} ({clip_cfg['pretrained']}) siap, {num_classes} kelas training")

## 2. Ekstrak embedding: gallery (train+val) vs query (test)

Transform = `clip_preprocess` (bawaan CLIP, bukan `build_eval_transforms`
ConvNeXt) -- deterministik, sama dipakai utk ketiga split.

In [ ]:
@torch.no_grad()
def extract_split(csv_key: str, desc: str):
    ds = WikiArtDataset(
        csv_path=cfg["data"][csv_key], image_dir=cfg["data"]["image_dir"],
        label_column=cfg["data"]["label_column"], label_to_idx=label_to_idx,
        transform=clip_preprocess, filename_column=cfg["data"]["filename_column"],
    )
    loader = DataLoader(ds, batch_size=clip_cfg["batch_size"], shuffle=False, num_workers=0)
    from tqdm import tqdm
    embs, labs = [], []
    for images, lbl in tqdm(loader, desc=desc, leave=False):
        feat = clip_model.encode_image(images.to(device))
        feat = torch.nn.functional.normalize(feat, dim=1)
        embs.append(feat.cpu().numpy()); labs.append(lbl.numpy())
    fnames = ds.df[cfg["data"]["filename_column"]].tolist()
    return np.concatenate(embs, axis=0), np.concatenate(labs, axis=0), fnames

train_emb, train_lbl, train_fn = extract_split("train_csv", "embed:train")
val_emb,   val_lbl,   val_fn   = extract_split("val_csv",   "embed:val")
test_emb,  test_lbl,  test_fn  = extract_split("test_csv",  "embed:test")

gallery_emb    = np.concatenate([train_emb, val_emb], axis=0)
gallery_labels = np.concatenate([train_lbl, val_lbl], axis=0)
gallery_fnames = train_fn + val_fn
query_emb, query_labels, query_fnames = test_emb, test_lbl, test_fn
print(f"gallery (train+val): {len(gallery_labels)} | query (test): {len(query_labels)}")

## 3. Recall@k / Precision@k / mAP@k vs baseline acak

In [ ]:
KS = (1, 5, 10)
metrics = evaluate_retrieval(query_emb, query_labels, gallery_emb, gallery_labels, ks=KS)
chance = {k: chance_recall_at_k(gallery_labels, k) for k in KS}

print(f"{'k':>4} | {'Recall@k':>10} | {'chance':>8} | {'Precision@k':>12} | {'mAP@k':>8}")
for k in KS:
    print(f"{k:4d} | {metrics[f'recall@{k}']:10.3f} | {chance[k]:8.3f} | "
          f"{metrics[f'precision@{k}']:12.3f} | {metrics[f'map@{k}']:8.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(KS)); w = 0.35
ax.bar(x - w/2, [metrics[f"recall@{k}"] for k in KS], w, label="CLIP (v3)")
ax.bar(x + w/2, [chance[k] for k in KS], w, label="chance (acak)")
ax.set_xticks(x, [f"Recall@{k}" for k in KS]); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Retrieval v3 (CLIP frozen): model vs baseline acak"); plt.tight_layout(); plt.show()

resolve_path("outputs/v3_clip/retrieval_metrics.json").write_text(
    json.dumps({"metrics": metrics, "chance": chance, "model": "v3_clip"}, indent=2),
    encoding="utf-8")

## 4. Contoh nyata: 1 query -> top-5 hasil

In [ ]:
rng = np.random.default_rng(cfg["seed"])
n_show = 4
q_idx = rng.choice(len(query_labels), size=n_show, replace=False)
idx, scores = nearest_neighbors(query_emb[q_idx], gallery_emb, top_k=5)

fig, axes = plt.subplots(n_show, 6, figsize=(18, 3.1 * n_show))
raw_dir = resolve_path(cfg["data"]["image_dir"])
from PIL import Image
for r, qi in enumerate(q_idx):
    qimg = Image.open(raw_dir / query_fnames[qi]).convert("RGB")
    axes[r, 0].imshow(qimg); axes[r, 0].set_title(f"QUERY\n{idx_to_label[query_labels[qi]]}", fontsize=9)
    axes[r, 0].axis("off")
    for c in range(5):
        gi = idx[r, c]
        gimg = Image.open(raw_dir / gallery_fnames[gi]).convert("RGB")
        ok = gallery_labels[gi] == query_labels[qi]
        axes[r, c+1].imshow(gimg)
        axes[r, c+1].set_title(f"#{c+1} sim={scores[r,c]:.2f}\n{idx_to_label[gallery_labels[gi]]}",
                               fontsize=8, color="green" if ok else "red")
        axes[r, c+1].axis("off")
fig.suptitle("Visual Search (v3 CLIP): query (kiri) -> top-5 hasil retrieval", fontsize=13)
plt.tight_layout(); plt.savefig(resolve_path("outputs/v3_clip/retrieval_examples.png"), dpi=100); plt.show()

## 5. Index embedding seluruh katalog

Sama seperti v1/v2: meliputi SEMUA gambar (`embedding.source_csv`), termasuk
kelas yang dibuang dari training (label -1).

In [ ]:
catalog_ds = WikiArtDataset(
    csv_path=cfg["embedding"]["source_csv"], image_dir=cfg["data"]["image_dir"],
    label_column=cfg["data"]["label_column"], label_to_idx=label_to_idx,
    transform=clip_preprocess, filename_column=cfg["data"]["filename_column"],
)
catalog_loader = DataLoader(catalog_ds, batch_size=clip_cfg["batch_size"], shuffle=False, num_workers=0)

from tqdm import tqdm
cat_embs, cat_labs = [], []
with torch.no_grad():
    for images, lbl in tqdm(catalog_loader, desc="embed:katalog", leave=False):
        feat = torch.nn.functional.normalize(clip_model.encode_image(images.to(device)), dim=1)
        cat_embs.append(feat.cpu().numpy()); cat_labs.append(lbl.numpy())
cat_emb = np.concatenate(cat_embs, axis=0)
cat_labels = np.concatenate(cat_labs, axis=0)
cat_fnames = catalog_ds.df[cfg["data"]["filename_column"]].tolist()

save_embedding_index(cat_emb, cat_fnames, cat_labels, clip_cfg["embedding_output_path"])
print(f"index katalog CLIP: {cat_emb.shape[0]} gambar x {cat_emb.shape[1]}-d -> "
      f"{clip_cfg['embedding_output_path']}")

## 6. Uji instance retrieval -- skenario "scan lukisan"

Sama seperti v2 section 6 -- simulasikan foto HP (distorsi ringan) dari
gambar katalog, cek apakah CLIP menemukan file ASLI-nya sbg hasil teratas.

In [ ]:
from PIL import Image

# CLIP butuh preprocessing-nya sendiri (clip_preprocess) -- TIDAK reuse
# src.transforms.build_capture_simulation_transform punya v2 (itu mengembalikan
# tensor ternormalisasi ImageNet, bukan PIL, jadi tidak bisa dirangkai dgn
# clip_preprocess). Distorsi "foto HP" ditulis ulang di sini secara manual
# (rotasi + color jitter + blur ringan), lalu diikuti clip_preprocess.
import torchvision.transforms as T
simulate_photo = T.Compose([
    T.RandomRotation(8), T.ColorJitter(brightness=0.25, contrast=0.2, saturation=0.2),
    T.GaussianBlur(3, sigma=(0.1, 1.0)),
])

raw_dir = resolve_path(cfg["data"]["image_dir"])
N_QUERY = 150
BATCH = 32
rng2 = np.random.default_rng(cfg["seed"])
query_idx = rng2.choice(len(cat_fnames), size=N_QUERY, replace=False)

sim_embs = []
with torch.no_grad():
    for start in tqdm(range(0, N_QUERY, BATCH), desc="simulasi foto HP"):
        batch_idx = query_idx[start:start + BATCH]
        imgs = [clip_preprocess(simulate_photo(Image.open(raw_dir / cat_fnames[qi]).convert("RGB")))
                for qi in batch_idx]
        x = torch.stack(imgs).to(device)
        f = torch.nn.functional.normalize(clip_model.encode_image(x), dim=1)
        sim_embs.append(f.cpu().numpy())
sim_embs = np.concatenate(sim_embs, axis=0)

idxs, scores = nearest_neighbors(sim_embs, cat_emb, top_k=10)
hit1  = idxs[:, 0] == query_idx
hit5  = (idxs[:, :5]  == query_idx[:, None]).any(axis=1)
hit10 = (idxs[:, :10] == query_idx[:, None]).any(axis=1)
print(f"Instance retrieval v3 CLIP (n={N_QUERY} simulasi 'foto HP'):")
print(f"  top-1 : {hit1.mean():.3f}")
print(f"  top-5 : {hit5.mean():.3f}")
print(f"  top-10: {hit10.mean():.3f}")

resolve_path("outputs/v3_clip/instance_retrieval_metrics.json").write_text(
    json.dumps({"n_query": N_QUERY, "top1": float(hit1.mean()),
               "top5": float(hit5.mean()), "top10": float(hit10.mean())}, indent=2),
    encoding="utf-8")

## 7. Catatan interpretasi (untuk laporan)

- Metodologi evaluasi (Recall@k, instance retrieval "scan lukisan") SENGAJA
  dibuat identik dgn `v2_22class/02_embedding_retrieval.ipynb` supaya hasil
  v2 vs v3 bisa dibandingkan langsung di `notebooks/04_compare_models.ipynb`.
- **Tidak ada tahap export ONNX/TorchScript di sini** -- v3 adalah baseline
  pembanding akademik, bukan kandidat deployment (`backend/` tetap pakai
  v2). Jangan overclaim "v3 siap produksi" tanpa langkah integrasi tambahan.
- Embedding CLIP (512-d, `ViT-B-32`) beda dimensi dari embedding v1/v2
  (768-d, `convnext_small`) -- TIDAK bisa dicampur di satu index/katalog yang
  sama, konsisten dgn prinsip "ruang vektor terpisah per model" di CLAUDE.md.